# Lab 8: Value at Risk and Its Backtests

*Financial Engineering in Practice*, Chapter 8.

**What you will explore:** how often losses exceed a 99% VaR, in calm markets and in crises, for six common models.

You will backtest six VaR models on daily U.S. market returns since 1930, count and test their breaches, push them into crises and deeper into the tail, and backtest expected shortfall, the measure that replaced VaR in the Basel rules for trading books. Before each step, write down a prediction.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import lab08

SOURCE = "french"     # "french" downloads daily factors; "sim" uses fake data to test the code
FF3_DAILY_FILE = None # e.g. "data/F-F_Research_Data_Factors_daily_CSV.zip"
OUT = Path("report")
OUT.mkdir(exist_ok=True)

d = lab08.load(SOURCE, FF3_DAILY_FILE)
SIM = d["simulated"]
fc = lab08.forecasts(d)   # about half a minute: GARCH models are re-estimated every year
print(fc["dates"][fc["first"]].date(), "to", fc["dates"][-1].date(), "| simulated:", SIM)

## Part A: 99% VaR backtests

Over about 24,000 trading days, a correct 99% VaR should be breached about 240 times.

**Predict first:** which models will be breached too often? Will any be breached too rarely?

In [ ]:
a = lab08.part_a(fc)
a["period"] = f"{fc['dates'][fc['first'] + 1].date()} to {fc['dates'][-1].date()}"
lab08.fig_path(fc, OUT, SIM)
lab08.fig_rates(a, OUT, SIM)
display(Image(OUT / "fig8_1_path.png"))
display(Image(OUT / "fig8_2_rates.png"))
a["table"][["model", "breaches", "expected", "rate", "kupiec_p", "independence_p", "cc_p", "share_years_red"]].round(4)

## Part B: Clustering

**Predict first:** if a model was breached yesterday, what is the chance it is breached today? For a correct model it is 1%.

In [ ]:
b = lab08.part_b(fc)
lab08.fig_per_year(b, OUT, SIM)
display(Image(OUT / "fig8_3_per_year.png"))
a["table"][["model", "p_breach_after_none", "p_breach_after_breach"]].round(4)

In [ ]:
b["worst_years"]

## Part C: Stress tests

**Predict first:** how many breaches would a correct 99% VaR produce in October 1987 (22 trading days)? How many will the models produce? Which models will hold up best at 99.9%?

In [ ]:
c = lab08.part_c(fc, d)
lab08.fig_levels(c, OUT, SIM)
display(Image(OUT / "fig8_4_levels.png"))
c["crises"].pivot_table(index="period", columns="model", values="breaches", sort=False)

In [ ]:
c["levels"].pivot_table(index="confidence", columns="model", values="ratio").round(2)

In [ ]:
c["windows"].round(4)  # historical simulation with 1, 2 and 4 years of data

## Part D: Expected shortfall at 97.5%

**Predict first:** a model whose VaR is breached too often will probably also understate ES. Will the ES backtest separate the models more or less clearly than the VaR backtest did?

In [ ]:
dd = lab08.part_d(fc)
lab08.fig_es(dd, OUT, SIM)
display(Image(OUT / "fig8_5_es.png"))
dd.round(3)

## Questions to think about

1. Which models deliver about 1% breaches at 99%? Which pass the independence test?
2. How do the models fare in crises and deeper in the tail (99.5%, 99.9%)?
3. Does expected shortfall fix the weaknesses of VaR, or inherit them?

Write one paragraph on what you saw, then compare it with the chapter's section Where the evidence stands.